In [9]:
# Model1: XGBoost model to predict Electrical Conductance (EC)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import r2_score

import optuna  # pip install optuna
# XGBoost (install first if needed: `pip install xgboost`)
import xgboost as xgb

In [10]:
# Load engineered training features and join with EC target

features_path = "../New Datasets/Combined/combined_training_engineered.csv"
water_quality_path = "../Provided Datasets/water_quality_training_dataset.csv"

combined = pd.read_csv(features_path)
water_quality = pd.read_csv(water_quality_path)

# Standardize join keys to match `combined`
water_quality_std = water_quality.rename(
    columns={
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Sample Date": "sample_date",
    }
)

# Keep only join keys + EC target
ec_target = water_quality_std[["latitude", "longitude", "sample_date", "Electrical Conductance"]]

# Inner join to align features with EC labels
full = combined.merge(ec_target, on=["latitude", "longitude", "sample_date"], how="inner")

print("Features shape (combined):", combined.shape)
print("Water quality shape:", water_quality.shape)
print("Joined training shape:", full.shape)
full.head()

Features shape (combined): (9319, 82)
Water quality shape: (9319, 6)
Joined training shape: (9319, 83)


,latitude,longitude,sample_date,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,gaia_recent_change_5y_frac,gaia_years_since_change_mean,gaia_transition_year_mean_changed_pixels,gsw_change,gsw_extent,...,hri,water_perm,water_instab,recurrence_ratio,seasonal_water,esa_change_intensity,wb_x_impervious,eci_x_impervious,gsw_occ_x_impervious,Electrical Conductance
0,-34.405833,19.600556,01-10-2014,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,4703.580299,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,1008.0
1,-34.405833,19.600556,02-08-2011,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,1984.043203,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,1237.0
2,-34.405833,19.600556,02-12-2015,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,6827.433216,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,1053.0
3,-34.405833,19.600556,03-07-2013,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,1466.619596,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,1167.0
4,-34.405833,19.600556,03-09-2014,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,2919.631339,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,877.0


In [11]:
# Build feature matrix X and target y for EC

# Columns to exclude from features
exclude_cols = {
    "Electrical Conductance",            # target
    "Total Alkalinity",                 # do not include
    "Dissolved Reactive Phosphorus",    # do not include
    "latitude", "longitude",           # do not include
    "sample_date",                      # string key, not a feature
}

feature_cols = [c for c in full.columns if c not in exclude_cols]
X = full[feature_cols]
y = full["Electrical Conductance"]

print("Number of features used:", len(feature_cols))
print("Example feature columns:", feature_cols[:10])

Number of features used: 79
Example feature columns: ['gaia_changed_ever_frac', 'gaia_impervious_frac_by_sample_year', 'gaia_recent_change_5y_frac', 'gaia_years_since_change_mean', 'gaia_transition_year_mean_changed_pixels', 'gsw_change', 'gsw_extent', 'gsw_occurrence', 'gsw_recurrence', 'gsw_seasonality']


In [12]:
# Train XGBoost model and report R² + feature importances

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

# R² scores
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Train R²: {r2_train:.3f}")
print(f"Test  R²: {r2_test:.3f}")

# Feature importances
importances = model.feature_importances_
fi = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi = fi.sort_values("importance", ascending=False)

print("\nTop 20 most important features for predicting EC:")
print(fi.head(20).to_string(index=False))

fi.head(20)

Train R²: 0.966
Test  R²: 0.861

Top 20 most important features for predicting EC:
                                 feature  importance
                    esa_flooded_frac_1km    0.063330
                              water_perm    0.061458
                     esa_forest_frac_1km    0.051288
                      esa_grass_frac_1km    0.050162
                      esa_water_frac_1km    0.040515
                        recurrence_ratio    0.040189
                     gsw_change_mean_1km    0.036312
                 gsw_recurrence_mean_1km    0.034517
                gsw_seasonality_mean_1km    0.032356
                                    soil    0.031948
                          esa_lccs_class    0.031014
                      esa_other_frac_1km    0.029847
                 esa_sparse_veg_frac_1km    0.029777
                 gsw_occurrence_mean_1km    0.029413
                      esa_shrub_frac_1km    0.027371
gaia_transition_year_mean_changed_pixels    0.025409
                

,feature,importance
44,esa_flooded_frac_1km,0.063330
71,water_perm,0.061458
38,esa_forest_frac_1km,0.051288
40,esa_grass_frac_1km,0.050162
37,esa_water_frac_1km,0.040515
73,recurrence_ratio,0.040189
50,gsw_change_mean_1km,0.036312
48,gsw_recurrence_mean_1km,0.034517
47,gsw_seasonality_mean_1km,0.032356
24,soil,0.031948


In [ ]:
# Stratified K-Fold + Optuna hyperparameter tuning for EC model

# Bin the continuous target into quantiles for stratification
n_bins = 10
# qcut can have duplicate bin edges; drop duplicates
y_strat = pd.qcut(y, q=n_bins, labels=False, duplicates='drop')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
        "random_state": 42,
        "n_jobs": -1,
    }

    model = xgb.XGBRegressor(**params)

    cv_scores = []
    for train_idx, valid_idx in skf.split(X, y_strat):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        model.fit(X_tr, y_tr)
        y_val_pred = model.predict(X_val)
        cv_scores.append(r2_score(y_val, y_val_pred))

    return float(np.mean(cv_scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best CV R²:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Train final model with best hyperparameters and report R² + feature importances

best_params = study.best_params.copy()
best_params.update({"random_state": 42, "n_jobs": -1})

final_model = xgb.XGBRegressor(**best_params)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

final_model.fit(X_train, y_train)

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Final model (with tuned params) Train R²: {r2_train:.3f}")
print(f"Final model (with tuned params) Test  R²: {r2_test:.3f}")

# Feature importances from tuned model
importances = final_model.feature_importances_
fi_tuned = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi_tuned = fi_tuned.sort_values("importance", ascending=False)

print("\nTop 20 most important features for predicting EC (tuned model):")
print(fi_tuned.head(20).to_string(index=False))

fi_tuned.head(20)